In [8]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")  # << 很多人的 LookupError 就是缺這個

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [1]:
# -*- coding: utf-8 -*-
"""
Research-grade ESG claim extraction (PDF -> CSV)

Modified goal
- Keep FULL SENTENCES
- A sentence is selected if it CONTAINS sustainability / environmental claim content
- The whole sentence does NOT need to be a pure claim sentence
- No short-claim exception: sentences below MIN_SENT_LEN are dropped
- Exclude common noise such as report headers, SDG text, table-like rows,
  nutrition/product-formula text, and financial / remuneration / board-related sentences
- Optionally keep only environmental claim-containing sentences
- Add sentence_id starting from sen_36

Outputs
- claims_research.csv
- bad_pdfs.csv
- runtime_report.csv
"""

import os
import re
import time
import csv
import shutil
import subprocess

import fitz  # PyMuPDF
import nltk
from tqdm import tqdm

# ---------------- NLTK safe init ----------------
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt", quiet=True)

# ========= 你要改的設定 =========
ROOT = "C:/greenwashingclean/reports"
TEST_FOLDER = None          # 測單一資料夾；跑全部 -> None  "2020_2020"
TOP_K = 2
MIN_CHAR_THRESHOLD = 1500          # 抽字太少：可能掃描pdf/抽字失敗

FORCE_RERUN = False                # True：忽略 done_set，全部重跑

# --- language / sentence controls ---
REMOVE_CHINESE = True
ENGLISH_ONLY = True
ENGLISH_RATIO_THRESHOLD = 0.60

MIN_SENT_LEN = 50
MAX_SENT_LEN = 450

# --- claim scope ---
ONLY_ENVIRONMENTAL_CLAIMS = True   # True：只保留 topic_guess 為 E / Mixed 的句子

# --- sentence id ---
START_SENTENCE_ID = 69             # 會從 sen_36 開始編

# --- selection controls ---
PAGE_BALANCE = True
MAX_PER_PAGE = 2
MAX_PER_SECTION = 5

# --- OCR fallback (optional) ---
OCR_FALLBACK = False
OCR_LANGUAGE = "eng"
# =================================

OUT_CLAIMS = "claims_research.csv"
OUT_BAD = "bad_pdfs.csv"
OUT_RUNTIME = "runtime_report.csv"

folder_pat = re.compile(r"^(?P<year>\d{4})_(?P<sic>\d{4})$")

# ---------------- ESG keywords by topic ----------------
E_KEYWORDS = [
    "sustain", "sustainability", "climate", "carbon", "co2", "co2e", "ghg", "emission",
    "net zero", "renew", "renewable", "recycl", "circular", "environment", "planet",
    "footprint", "energy", "water", "waste", "biodivers", "packaging",
    "material", "decarbon", "pollution", "deforest", "nature", "green", "eco"
]
S_KEYWORDS = [
    "human rights", "labor", "labour", "worker", "wage", "salary",
    "health and safety", "safety", "injury", "divers", "inclusion", "equity",
    "community", "training", "wellbeing", "harassment", "child labor",
    "forced labor", "modern slavery", "supplier", "supply chain"
]
G_KEYWORDS = [
    "governance", "compliance", "ethic", "anti-corruption", "bribery",
    "transparen", "audit", "board", "risk management", "whistle", "integrity",
    "policy", "code of conduct", "due diligence", "data privacy", "cyber"
]

ALL_ESG = sorted(set(E_KEYWORDS + S_KEYWORDS + G_KEYWORDS), key=len, reverse=True)
kw_pattern = re.compile(r"(" + "|".join(re.escape(k) for k in ALL_ESG) + r")", re.IGNORECASE)

# “claim-like” verbs / phrases
CLAIM_VERBS = [
    "aim", "commit", "pledge", "target", "will", "plan", "reduce", "achieve",
    "increase", "improve", "deliver", "ensure", "support", "transition",
    "offset", "eliminate", "phase out", "set out", "strive", "help", "enable",
    "replace", "reuse", "recycle", "refill", "source", "lower", "cut"
]
verb_pattern = re.compile(r"\b(" + "|".join(re.escape(v) for v in CLAIM_VERBS) + r")\b", re.IGNORECASE)

CLAIM_PHRASES = [
    "aim to", "plan to", "commit to", "target to", "pledge to",
    "set a target", "set targets", "science-based target", "aligned with",
    "by 2030", "by 2040", "by 2050", "working toward", "working towards"
]
phrase_pattern = re.compile(r"(" + "|".join(re.escape(p) for p in CLAIM_PHRASES) + r")", re.IGNORECASE)

# green marketing / slogan-like claim content
GREEN_CLAIM_WORDS = [
    "eco friendly", "environmentally friendly", "better for the planet",
    "good for the planet", "kinder on the planet", "sustainable future",
    "greener", "green choice", "conscious choice", "planet friendly",
    "future proof", "future-proof", "climate positive", "carbon negative",
    "carbon neutral", "positive planet", "cleaner energy", "healthier planet",
    "nature positive", "low carbon", "lower carbon", "better for the environment",
    "world's first carbon negative", "world's first carbon neutral",
    "environmental airline of the year", "for the planet"
]
green_claim_pattern = re.compile(
    r"(" + "|".join(re.escape(k) for k in GREEN_CLAIM_WORDS) + r")",
    re.IGNORECASE
)

# material / packaging / circularity claim content
MATERIAL_CLAIM_WORDS = [
    "recycled plastic", "recycled materials", "renewable materials",
    "recyclable", "recycled", "renewable", "rpet", "fsc", "certified",
    "refillable", "plant based", "plant-based", "organic", "sustainably sourced",
    "circular", "reuse", "reusable", "refill", "second life", "take back",
    "take-back", "packaging", "compostable", "bio-based", "biomass", "biodegradable"
]
material_claim_pattern = re.compile(
    r"(" + "|".join(re.escape(k) for k in MATERIAL_CLAIM_WORDS) + r")",
    re.IGNORECASE
)

# section weak labels
SECTIONS = {
    "Strategy": ["strategy","ceo message","letter from","vision","roadmap","foreword","our approach","our commitment"],
    "Targets": ["target","targets","goal","goals","commitment","commitments","2030","2040","2050","science based","sbti","pathway"],
    "Environment": ["environment","climate","carbon","emissions","energy","water","waste","net zero","decarbon","biodiversity","circular"],
    "Performance": ["performance","performance data","metrics","kpi","sasb","tcfd","gri","data table","appendix"]
}

# evidence / quality patterns
RE_YEAR = re.compile(r"\b(19|20)\d{2}\b")
RE_BY_YEAR = re.compile(r"\bby\b\s*(19|20)\d{2}\b", re.IGNORECASE)
RE_PERCENT = re.compile(r"(\d+(\.\d+)?\s*%|\bpercent\b)", re.IGNORECASE)
RE_NUMBER = re.compile(r"\d")
RE_SCOPE = re.compile(r"\bscope\s*[123]\b", re.IGNORECASE)
RE_SBTI = re.compile(r"\b(sbti|science[-\s]?based)\b", re.IGNORECASE)
RE_NETZERO = re.compile(r"\bnet\s*zero\b|\bcarbon\s*neutral\b|\bcarbon\s*negative\b|\bclimate\s*positive\b|\bclimate\s*neutral\b", re.IGNORECASE)
RE_KPI = re.compile(r"\b(kpi|metric|metrics|baseline|progress|gri|sasb|tcfd)\b", re.IGNORECASE)
RE_MATERIAL = re.compile(r"\b(renewable|recycled|recyclable|rpet|fsc|certified|refillable|plant[-\s]?based|organic|bio-based|biomass|biodegradable|compostable)\b", re.IGNORECASE)
RE_CARBON = re.compile(r"\b(carbo[n]?|co2|co2e|ghg|emission|emissions|decarbon)\b", re.IGNORECASE)
RE_TARGET = re.compile(r"\b(target|goal|goals|commit|commitment|pledge|plan|roadmap|ambition|will|aim)\b", re.IGNORECASE)
RE_CIRCULAR = re.compile(r"\b(circular|recycle|recycled|recyclable|reuse|reusable|refill|second life|take back|take-back)\b", re.IGNORECASE)

# dehyphenation
RE_HYPHEN_BREAK = re.compile(r"(\w)-\s+(\w)")

# Chinese (CJK) characters
RE_CJK = re.compile(r"[\u4e00-\u9fff]")

# noise
RE_URL = re.compile(r"(https?://|www\.|\.com\b|\.org\b|\.net\b)", re.IGNORECASE)
RE_EMAIL = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
RE_PAGE_ONLY = re.compile(r"^\s*(page\s*\d+|\d+)\s*$", re.IGNORECASE)

def contains_chinese(s: str) -> bool:
    return bool(RE_CJK.search(s))

def parse_pdf_filename(filename: str):
    stem = filename.rsplit(".", 1)[0]
    parts = stem.split("_")
    if len(parts) < 2:
        return stem, "", ""
    report_year = parts[-1]

    if len(parts) >= 3:
        possible_ticker = parts[-2]
        if re.fullmatch(r"(?=.*[A-Za-z])[A-Za-z0-9]{1,10}", possible_ticker):
            return "_".join(parts[:-2]), possible_ticker.upper(), report_year

    return "_".join(parts[:-1]), "", report_year

def ensure_header(path: str, fieldnames):
    if os.path.exists(path):
        return
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writeheader()

def load_done_set(out_csv_path: str):
    done = set()
    if not os.path.exists(out_csv_path):
        return done
    try:
        with open(out_csv_path, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                folder = row.get("source_folder", "")
                file = row.get("file", "")
                if folder and file:
                    done.add((folder, file))
    except Exception:
        return set()
    return done

def clean_sentence(s: str) -> str:
    # 1) Unicode normalization-like replacements
    replacements = {
        "\u00ad": "",     # soft hyphen
        "\xa0": " ",      # non-breaking space
        "\ufeff": " ",    # BOM
        "\u200b": " ",    # zero-width space
        "\u200c": " ",
        "\u200d": " ",
        "\u2060": " ",
        "“": '"', "”": '"',
        "‘": "'", "’": "'",
        "–": "-", "—": "-", "−": "-", "-": "-", "‒": "-",
        "ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl",
        "％": "%", "／": "/", "（": "(", "）": ")", "：": ":",
    }
    for old, new in replacements.items():
        s = s.replace(old, new)

    # 2) Remove control chars
    s = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", s)

    # 3) Fix PDF line-break hyphens
    s = RE_HYPHEN_BREAK.sub(r"\1\2", s)

    # 4) Remove bullet / icon symbols
    bad_chars = [
        "•", "·", "▪", "▫", "■", "□", "●", "○", "◦", "‣", "∙",
        "▶", "►", "◄", "▲", "△", "▼", "▽", "◆", "◇", "★", "☆", "※"
    ]
    for ch in bad_chars:
        s = s.replace(ch, " ")

    # 5) Remove trademark / copyright clutter
    for ch in ["™", "®", "©"]:
        s = s.replace(ch, "")

    # 6) Keep only useful characters for ESG text
    s = re.sub(r"[^A-Za-z0-9\s%.,:;()\[\]\"'&+\-/]", " ", s)

    # 7) Normalize slash spacing
    s = re.sub(r"\s*/\s*", "/", s)

    # 8) Compress whitespace
    s = re.sub(r"\s+", " ", s).strip()

    return s

def is_mostly_english(s: str, threshold=0.60) -> bool:
    ss = re.sub(r"\s+", "", s)
    if not ss:
        return False
    letters = re.findall(r"[A-Za-z]", ss)
    if not letters:
        return False
    ratio = len(letters) / len(ss)
    return ratio >= threshold

def is_noise_sentence(s: str) -> bool:
    s_low = s.lower().strip()

    if not s_low:
        return True
    if RE_PAGE_ONLY.fullmatch(s_low):
        return True
    if RE_URL.search(s_low) or RE_EMAIL.search(s_low):
        return True
    if "copyright" in s_low or "all rights reserved" in s_low:
        return True
    if s_low.count("|") >= 2:
        return True
    if len(re.findall(r"[A-Za-z]", s_low)) < 5:
        return True
    if len(s_low.split()) < 3:
        return True
    if re.fullmatch(r"[\W\d_]+", s_low):
        return True

    digit_ratio = sum(ch.isdigit() for ch in s_low) / max(len(s_low), 1)
    if digit_ratio > 0.45 and not RE_PERCENT.search(s_low):
        return True

    return False

def is_table_like_row(s: str) -> bool:
    s_low = s.lower()

    year_count = len(re.findall(r"\bfy(?:19|20)\d{2}\b|\b(?:19|20)\d{2}\b", s_low))
    percent_count = len(re.findall(r"\d+(?:\.\d+)?\s*%", s_low))
    number_count = len(re.findall(r"\d+(?:,\d+)?(?:\.\d+)?", s_low))

    if year_count >= 3:
        return True
    if percent_count >= 3:
        return True
    if number_count >= 8 and len(s_low.split()) < 50:
        return True

    return False

def is_nutrition_or_product_formula_noise(s: str) -> bool:
    s_low = s.lower()

    bad_terms = [
        "dha", "choline", "selenium", "manganese", "prebiotic",
        "infant milk powder", "mama formula", "maternal mothers milk powder",
        "reduced fat", "nutritional combination", "seven key nutrients",
        "product formula", "nutrition from natural sources", "algae oil"
    ]

    return any(t in s_low for t in bad_terms)

def is_metadata_or_sdg_or_financial_noise(s: str) -> bool:
    s_low = s.lower().strip()

    header_terms = [
        "sustainability report",
        "sustainability data book",
        "non-financial report",
        "financial and non-financial highlights",
        "strategy and targets",
        "our people metric",
        "food quality and safety close up",
        "introduction fy",
        "our supply chain",
        "packaging for a better planet",
        "sustainability vision",
        "conceptual diagram",
    ]
    if any(t in s_low for t in header_terms):
        return True

    admin_terms = [
        "the materials in this report were provided",
        "may not be used for commercial purpose",
        "is the eighth non-financial report issued",
        "key performance indicator",
        "kpi (key performance indicator)",
    ]
    if any(t in s_low for t in admin_terms):
        return True

    if re.match(r"^\d+\.\d+\s+by\s+2030", s_low):
        return True
    if "sustainable development goals" in s_low:
        return True
    if "united nations sustainable development summit" in s_low:
        return True
    if "approved at the united nations" in s_low:
        return True
    if re.match(r"^\d+\s+take urgent action to combat climate change", s_low):
        return True

    finance_terms = [
        "roe", "ebitda", "dividend", "equity ratio", "free cash flow",
        "board of directors", "executive board", "base salary",
        "short-term bonus", "long-term incentives", "remuneration",
        "attendance at the board meetings", "capital expenditures",
        "net sales", "operating profit", "payout ratio", "organic growth",
        "profitability", "annual base salary", "performance rights",
        "grant value", "board year", "compensation for 2020", "shares equivalent",
        "ceo:", "executive vice president", "financial stability",
        "medium-term management plan", "long-term vision", "future creation company",
        "telecommuting program", "pension plan", "additional fees", "audit",
        "committee meetings", "board reserves", "directors believe",
        "accounting records", "inventories", "nomination and sustainability committee"
    ]
    if any(t in s_low for t in finance_terms):
        return True

    if len(re.findall(r"\b(?:19|20)\d{2}\b", s_low)) >= 3:
        return True

    if "fy201" in s_low and "%" in s_low and len(s_low.split()) < 30:
        return True

    return False

def contains_claim_content(s: str) -> bool:
    s_low = s.lower()

    has_esg = bool(kw_pattern.search(s_low))
    has_claim_cue = bool(verb_pattern.search(s_low) or phrase_pattern.search(s_low))
    has_green = bool(green_claim_pattern.search(s_low))
    has_material = bool(material_claim_pattern.search(s_low))
    has_netzero = bool(RE_NETZERO.search(s_low))
    has_quant_env = bool(
        RE_PERCENT.search(s_low) and (
            RE_CARBON.search(s_low) or
            RE_MATERIAL.search(s_low) or
            "planet" in s_low or
            "environment" in s_low or
            "water" in s_low or
            "waste" in s_low or
            "energy" in s_low
        )
    )

    if has_green or has_material or has_netzero or has_quant_env:
        return True
    if has_esg and has_claim_cue:
        return True

    return False

def section_guess_from_text(page_text: str) -> str:
    lines = [ln.strip() for ln in page_text.splitlines() if ln.strip()]
    head = " ".join(lines[:10]).lower()
    for sec in ["Strategy", "Targets", "Environment", "Performance"]:
        for w in SECTIONS.get(sec, []):
            if w in head:
                return sec
    return "Other"

def topic_guess(sentence: str) -> str:
    s = sentence.lower()
    e = any(k in s for k in E_KEYWORDS)
    so = any(k in s for k in S_KEYWORDS)
    g = any(k in s for k in G_KEYWORDS)
    if e and not so and not g:
        return "E"
    if so and not e and not g:
        return "S"
    if g and not e and not so:
        return "G"
    if e or so or g:
        return "Mixed"
    return "Other"

def claim_type_guess(s: str) -> str:
    s_low = s.lower()

    if (RE_PERCENT.search(s_low) or RE_NUMBER.search(s_low)) and (RE_BY_YEAR.search(s_low) or RE_TARGET.search(s_low)):
        return "target_commitment"
    if RE_NETZERO.search(s_low) or RE_CARBON.search(s_low):
        return "carbon_claim"
    if RE_CIRCULAR.search(s_low) or RE_MATERIAL.search(s_low) or material_claim_pattern.search(s_low):
        return "recycling_circularity"
    if green_claim_pattern.search(s_low):
        return "marketing_claim"
    if RE_PERCENT.search(s_low) or RE_NUMBER.search(s_low):
        return "quantitative_claim"
    if RE_TARGET.search(s_low):
        return "target_commitment"
    if contains_claim_content(s_low):
        return "general_green_claim"
    return "other"

def evidence_flags(s: str) -> dict:
    return {
        "has_number": int(bool(RE_NUMBER.search(s))),
        "has_year": int(bool(RE_YEAR.search(s))),
        "has_by_year": int(bool(RE_BY_YEAR.search(s))),
        "has_percent": int(bool(RE_PERCENT.search(s))),
        "has_scope": int(bool(RE_SCOPE.search(s))),
        "has_sbti": int(bool(RE_SBTI.search(s))),
        "has_netzero": int(bool(RE_NETZERO.search(s))),
        "has_kpi": int(bool(RE_KPI.search(s))),
        "has_material": int(bool(RE_MATERIAL.search(s))),
        "sentence_type": claim_type_guess(s),
        "has_green_marketing": int(bool(green_claim_pattern.search(s))),
    }

def score_sentence(s: str) -> int:
    score = 0
    s_low = s.lower()

    if contains_claim_content(s):
        score += 4

    if verb_pattern.search(s):
        score += 2
    if phrase_pattern.search(s):
        score += 2

    if green_claim_pattern.search(s):
        score += 3

    if material_claim_pattern.search(s):
        score += 2

    if RE_NUMBER.search(s):
        score += 2
    if RE_YEAR.search(s):
        score += 2
    if RE_BY_YEAR.search(s):
        score += 3
    if RE_PERCENT.search(s):
        score += 3
    if RE_SCOPE.search(s):
        score += 3
    if RE_SBTI.search(s):
        score += 3
    if RE_NETZERO.search(s):
        score += 3
    if RE_KPI.search(s):
        score += 2
    if RE_MATERIAL.search(s):
        score += 2

    ct = claim_type_guess(s)
    if ct == "target_commitment":
        score += 2
    elif ct == "quantitative_claim":
        score += 2
    elif ct == "carbon_claim":
        score += 2
    elif ct == "marketing_claim":
        score += 1
    elif ct == "recycling_circularity":
        score += 1

    if len(s) < 35:
        score -= 4
    elif len(s) < 50:
        score -= 1
    if len(s) > 380:
        score -= 2

    vague = [
        "we believe", "we strive", "we support", "our mission", "our purpose",
        "better future", "we care", "we are committed to sustainability",
        "for a better world"
    ]
    if any(v in s_low for v in vague):
        if not (
            RE_NUMBER.search(s) or RE_PERCENT.search(s) or RE_BY_YEAR.search(s)
            or green_claim_pattern.search(s)
        ):
            score -= 2

    return score

def normalize_sentence(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^\w\s%]", "", s)
    s = re.sub(r"\b(the|a|an|our|we|is|are|to|of|and|for|in)\b", " ", s)
    s = re.sub(r"\s+", "", s)
    return s

def extract_pages_pymupdf(pdf_path: str):
    pages = []
    doc = fitz.open(pdf_path)
    for i in range(doc.page_count):
        txt = doc.load_page(i).get_text("text") or ""
        pages.append((i + 1, txt))
    doc.close()
    return pages

def try_ocr(pdf_path: str) -> str:
    if not OCR_FALLBACK:
        return pdf_path
    if shutil.which("ocrmypdf") is None:
        return pdf_path

    ocr_out = pdf_path.rsplit(".", 1)[0] + "_OCR.pdf"
    if os.path.exists(ocr_out):
        return ocr_out

    try:
        cmd = ["ocrmypdf", "--skip-text", "-l", OCR_LANGUAGE, pdf_path, ocr_out]
        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if os.path.exists(ocr_out):
            return ocr_out
    except Exception:
        return pdf_path

    return pdf_path

def balanced_topk(items, k, max_per_page=2, max_per_section=5):
    out = []
    page_ct = {}
    sec_ct = {}

    for it in items:
        sc, pg, sec = it[0], it[1], it[2]
        if max_per_page is not None and page_ct.get(pg, 0) >= max_per_page:
            continue
        if max_per_section is not None and sec_ct.get(sec, 0) >= max_per_section:
            continue
        out.append(it)
        page_ct[pg] = page_ct.get(pg, 0) + 1
        sec_ct[sec] = sec_ct.get(sec, 0) + 1
        if len(out) >= k:
            break
    return out

# ---------------- Output schema ----------------
CLAIMS_FIELDS = [
    "sentence_id",
    "folder_year","sic","company","ticker","report_year",
    "source_folder","file",
    "rank","score","page","section_guess","topic_guess","sentence_type",
    "has_number","has_year","has_by_year","has_percent","has_scope","has_sbti","has_netzero","has_kpi","has_material","has_green_marketing",
    "sentence"
]
BAD_FIELDS = [
    "source_folder","folder_year","sic","file",
    "company","ticker","report_year",
    "char_count","status","error"
]
RUNTIME_FIELDS = [
    "source_folder","folder_year","sic","file",
    "company","ticker","report_year",
    "secs","char_count","candidate_sentences","written_topk","avg_score_topk"
]

ensure_header(OUT_CLAIMS, CLAIMS_FIELDS)
ensure_header(OUT_BAD, BAD_FIELDS)
ensure_header(OUT_RUNTIME, RUNTIME_FIELDS)

done_set = set() if FORCE_RERUN else load_done_set(OUT_CLAIMS)

# ---------------- Collect jobs ----------------
pdf_jobs = []
folders = [TEST_FOLDER] if TEST_FOLDER else sorted(os.listdir(ROOT))

for folder_name in folders:
    if not folder_name:
        continue
    folder_path = os.path.join(ROOT, folder_name)
    if not os.path.isdir(folder_path):
        continue
    m = folder_pat.match(folder_name)
    if not m:
        continue
    folder_year, sic = m.group("year"), m.group("sic")
    for fn in sorted(os.listdir(folder_path)):
        if fn.lower().endswith(".pdf"):
            if FORCE_RERUN or (folder_name, fn) not in done_set:
                pdf_jobs.append((folder_name, folder_year, sic, fn, os.path.join(folder_path, fn)))

print("Already done PDFs:", 0 if FORCE_RERUN else len(done_set))
print("Remaining PDFs:", len(pdf_jobs))

total_written = 0
sentence_counter = START_SENTENCE_ID

with open(OUT_CLAIMS, "a", newline="", encoding="utf-8-sig") as f_out, \
     open(OUT_BAD, "a", newline="", encoding="utf-8-sig") as f_bad, \
     open(OUT_RUNTIME, "a", newline="", encoding="utf-8-sig") as f_run:

    w_out = csv.DictWriter(f_out, fieldnames=CLAIMS_FIELDS)
    w_bad = csv.DictWriter(f_bad, fieldnames=BAD_FIELDS)
    w_run = csv.DictWriter(f_run, fieldnames=RUNTIME_FIELDS)

    for (folder_name, folder_year, sic, fn, pdf_path) in tqdm(pdf_jobs, desc="Processing PDFs", unit="pdf"):
        t0 = time.time()
        company, ticker, report_year = parse_pdf_filename(fn)

        pdf_to_read = try_ocr(pdf_path)

        try:
            pages = extract_pages_pymupdf(pdf_to_read)
        except Exception as e:
            w_bad.writerow({
                "source_folder": folder_name, "folder_year": folder_year, "sic": sic,
                "file": fn, "company": company, "ticker": ticker, "report_year": report_year,
                "char_count": 0, "status": "ERROR_OPEN_OR_READ", "error": str(e)[:300]
            })
            w_run.writerow({
                "source_folder": folder_name, "folder_year": folder_year, "sic": sic,
                "file": fn, "company": company, "ticker": ticker, "report_year": report_year,
                "secs": round(time.time() - t0, 2), "char_count": 0,
                "candidate_sentences": 0, "written_topk": 0, "avg_score_topk": 0
            })
            continue

        char_count = sum(len(txt) for _, txt in pages if txt)

        if char_count < MIN_CHAR_THRESHOLD:
            w_bad.writerow({
                "source_folder": folder_name, "folder_year": folder_year, "sic": sic,
                "file": fn, "company": company, "ticker": ticker, "report_year": report_year,
                "char_count": char_count, "status": "LOW_TEXT_POSSIBLE_SCAN_OR_EXTRACT_FAIL", "error": ""
            })
            w_run.writerow({
                "source_folder": folder_name, "folder_year": folder_year, "sic": sic,
                "file": fn, "company": company, "ticker": ticker, "report_year": report_year,
                "secs": round(time.time() - t0, 2), "char_count": char_count,
                "candidate_sentences": 0, "written_topk": 0, "avg_score_topk": 0
            })
            continue

        candidates = []

        # Pass 1
        for page_num, page_text in pages:
            if not page_text:
                continue
            sec = section_guess_from_text(page_text)

            for s in nltk.sent_tokenize(page_text):
                s = clean_sentence(s)
                if not s:
                    continue

                if is_noise_sentence(s):
                    continue
                if is_table_like_row(s):
                    continue
                if is_nutrition_or_product_formula_noise(s):
                    continue
                if is_metadata_or_sdg_or_financial_noise(s):
                    continue
                if REMOVE_CHINESE and contains_chinese(s):
                    continue
                if len(s) < MIN_SENT_LEN or len(s) > MAX_SENT_LEN:
                    continue
                if ENGLISH_ONLY and not is_mostly_english(s, ENGLISH_RATIO_THRESHOLD):
                    continue
                if not contains_claim_content(s):
                    continue

                tg = topic_guess(s)
                if ONLY_ENVIRONMENTAL_CLAIMS and tg not in ("E", "Mixed"):
                    continue

                sc = score_sentence(s)
                flags = evidence_flags(s)
                candidates.append((sc, page_num, sec, s, flags))

        # Pass 2
        if len(candidates) < TOP_K:
            relaxed_candidates = []
            for page_num, page_text in pages:
                if not page_text:
                    continue
                sec = section_guess_from_text(page_text)

                for s in nltk.sent_tokenize(page_text):
                    s = clean_sentence(s)
                    if not s:
                        continue

                    if is_noise_sentence(s):
                        continue
                    if is_table_like_row(s):
                        continue
                    if is_nutrition_or_product_formula_noise(s):
                        continue
                    if is_metadata_or_sdg_or_financial_noise(s):
                        continue
                    if REMOVE_CHINESE and contains_chinese(s):
                        continue
                    if len(s) < MIN_SENT_LEN or len(s) > MAX_SENT_LEN:
                        continue
                    if ENGLISH_ONLY and not is_mostly_english(s, ENGLISH_RATIO_THRESHOLD):
                        continue

                    relaxed_hit = (
                        kw_pattern.search(s) or
                        green_claim_pattern.search(s) or
                        material_claim_pattern.search(s)
                    )
                    if not relaxed_hit:
                        continue

                    tg = topic_guess(s)
                    if ONLY_ENVIRONMENTAL_CLAIMS and tg not in ("E", "Mixed"):
                        continue

                    sc = score_sentence(s) - 1
                    flags = evidence_flags(s)
                    relaxed_candidates.append((sc, page_num, sec, s, flags))

            if relaxed_candidates:
                candidates.extend(relaxed_candidates)

        # Deduplicate
        best = {}
        for sc, pg, sec, sent, flags in candidates:
            norm = normalize_sentence(sent)
            if norm not in best or sc > best[norm][0]:
                best[norm] = (sc, pg, sec, sent, flags)

        uniq = list(best.values())
        uniq.sort(key=lambda x: x[0], reverse=True)

        # Top-k
        if PAGE_BALANCE:
            top = balanced_topk(
                [(sc, pg, sec, sent, flags) for (sc, pg, sec, sent, flags) in uniq],
                TOP_K,
                max_per_page=MAX_PER_PAGE,
                max_per_section=MAX_PER_SECTION
            )
        else:
            top = [(sc, pg, sec, sent, flags) for (sc, pg, sec, sent, flags) in uniq[:TOP_K]]

        for r, (sc, pg, sec, sent, flags) in enumerate(top, start=1):
            tg = topic_guess(sent)
            row = {
                "sentence_id": f"sen_{sentence_counter}",
                "folder_year": folder_year,
                "sic": sic,
                "company": company,
                "ticker": ticker,
                "report_year": report_year,
                "source_folder": folder_name,
                "file": fn,
                "rank": r,
                "score": sc,
                "page": pg,
                "section_guess": sec,
                "topic_guess": tg,
                "sentence_type": flags["sentence_type"],
                "has_number": flags["has_number"],
                "has_year": flags["has_year"],
                "has_by_year": flags["has_by_year"],
                "has_percent": flags["has_percent"],
                "has_scope": flags["has_scope"],
                "has_sbti": flags["has_sbti"],
                "has_netzero": flags["has_netzero"],
                "has_kpi": flags["has_kpi"],
                "has_material": flags["has_material"],
                "has_green_marketing": flags["has_green_marketing"],
                "sentence": sent
            }
            w_out.writerow(row)
            total_written += 1
            sentence_counter += 1

        avg_score_topk = round(sum(x[0] for x in top) / len(top), 2) if top else 0

        w_run.writerow({
            "source_folder": folder_name, "folder_year": folder_year, "sic": sic,
            "file": fn, "company": company, "ticker": ticker, "report_year": report_year,
            "secs": round(time.time() - t0, 2),
            "char_count": char_count,
            "candidate_sentences": len(uniq),
            "written_topk": len(top),
            "avg_score_topk": avg_score_topk
        })

print("DONE ✅  New sentences written:", total_written)
print("Outputs:", OUT_CLAIMS, OUT_BAD, OUT_RUNTIME)

Already done PDFs: 0
Remaining PDFs: 237


Processing PDFs: 100%|██████████| 237/237 [04:50<00:00,  1.22s/pdf]

DONE ✅  New sentences written: 472
Outputs: claims_research.csv bad_pdfs.csv runtime_report.csv


In [2]:
# -*- coding: utf-8 -*-
"""
Research-grade ESG claim extraction (PDF -> CSV)

Modified goal
- Keep FULL SENTENCES
- A sentence is selected if it CONTAINS sustainability / environmental claim content
- The whole sentence does NOT need to be a pure claim sentence
- No short-claim exception: sentences below MIN_SENT_LEN are dropped
- Exclude common noise such as report headers, SDG text, table-like rows,
  nutrition/product-formula text, and financial / remuneration / board-related sentences
- Optionally keep only environmental claim-containing sentences
- Add sentence_id starting from sen_69

Outputs
- claims_research.csv
- bad_pdfs.csv
- runtime_report.csv

New additions
- Keep original sentence in "sentence"
- Add lowercase version in "sentence_lower" for LLM / ML use
- Normalize ALL-CAPS extracted heading-like sentences before writing
"""

import os
import re
import time
import csv
import shutil
import subprocess

import fitz  # PyMuPDF
import nltk
from tqdm import tqdm

# ---------------- NLTK safe init ----------------
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt", quiet=True)

# ========= 你要改的設定 =========
ROOT = "C:/greenwashingclean/reports"
TEST_FOLDER = None          # 測單一資料夾；跑全部 -> None  "2020_2020"
TOP_K = 2
MIN_CHAR_THRESHOLD = 1500   # 抽字太少：可能掃描pdf/抽字失敗

FORCE_RERUN = False         # True：忽略 done_set，全部重跑

# --- language / sentence controls ---
REMOVE_CHINESE = True
ENGLISH_ONLY = True
ENGLISH_RATIO_THRESHOLD = 0.60

MIN_SENT_LEN = 50
MAX_SENT_LEN = 450

# --- claim scope ---
ONLY_ENVIRONMENTAL_CLAIMS = True   # True：只保留 topic_guess 為 E / Mixed 的句子

# --- sentence id ---
START_SENTENCE_ID = 69

# --- selection controls ---
PAGE_BALANCE = True
MAX_PER_PAGE = 2
MAX_PER_SECTION = 5

# --- OCR fallback (optional) ---
OCR_FALLBACK = False
OCR_LANGUAGE = "eng"
# =================================

OUT_CLAIMS = "claims_research.csv"
OUT_BAD = "bad_pdfs.csv"
OUT_RUNTIME = "runtime_report.csv"

folder_pat = re.compile(r"^(?P<year>\d{4})_(?P<sic>\d{4})$")

# ---------------- ESG keywords by topic ----------------
E_KEYWORDS = [
    "sustain", "sustainability", "climate", "carbon", "co2", "co2e", "ghg", "emission",
    "net zero", "renew", "renewable", "recycl", "circular", "environment", "planet",
    "footprint", "energy", "water", "waste", "biodivers", "packaging",
    "material", "decarbon", "pollution", "deforest", "nature", "green", "eco"
]
S_KEYWORDS = [
    "human rights", "labor", "labour", "worker", "wage", "salary",
    "health and safety", "safety", "injury", "divers", "inclusion", "equity",
    "community", "training", "wellbeing", "harassment", "child labor",
    "forced labor", "modern slavery", "supplier", "supply chain"
]
G_KEYWORDS = [
    "governance", "compliance", "ethic", "anti-corruption", "bribery",
    "transparen", "audit", "board", "risk management", "whistle", "integrity",
    "policy", "code of conduct", "due diligence", "data privacy", "cyber"
]

ALL_ESG = sorted(set(E_KEYWORDS + S_KEYWORDS + G_KEYWORDS), key=len, reverse=True)
kw_pattern = re.compile(r"(" + "|".join(re.escape(k) for k in ALL_ESG) + r")", re.IGNORECASE)

# “claim-like” verbs / phrases
CLAIM_VERBS = [
    "aim", "commit", "pledge", "target", "will", "plan", "reduce", "achieve",
    "increase", "improve", "deliver", "ensure", "support", "transition",
    "offset", "eliminate", "phase out", "set out", "strive", "help", "enable",
    "replace", "reuse", "recycle", "refill", "source", "lower", "cut"
]
verb_pattern = re.compile(r"\b(" + "|".join(re.escape(v) for v in CLAIM_VERBS) + r")\b", re.IGNORECASE)

CLAIM_PHRASES = [
    "aim to", "plan to", "commit to", "target to", "pledge to",
    "set a target", "set targets", "science-based target", "aligned with",
    "by 2030", "by 2040", "by 2050", "working toward", "working towards"
]
phrase_pattern = re.compile(r"(" + "|".join(re.escape(p) for p in CLAIM_PHRASES) + r")", re.IGNORECASE)

# green marketing / slogan-like claim content
GREEN_CLAIM_WORDS = [
    "eco friendly", "environmentally friendly", "better for the planet",
    "good for the planet", "kinder on the planet", "sustainable future",
    "greener", "green choice", "conscious choice", "planet friendly",
    "future proof", "future-proof", "climate positive", "carbon negative",
    "carbon neutral", "positive planet", "cleaner energy", "healthier planet",
    "nature positive", "low carbon", "lower carbon", "better for the environment",
    "world's first carbon negative", "world's first carbon neutral",
    "environmental airline of the year", "for the planet"
]
green_claim_pattern = re.compile(
    r"(" + "|".join(re.escape(k) for k in GREEN_CLAIM_WORDS) + r")",
    re.IGNORECASE
)

# material / packaging / circularity claim content
MATERIAL_CLAIM_WORDS = [
    "recycled plastic", "recycled materials", "renewable materials",
    "recyclable", "recycled", "renewable", "rpet", "fsc", "certified",
    "refillable", "plant based", "plant-based", "organic", "sustainably sourced",
    "circular", "reuse", "reusable", "refill", "second life", "take back",
    "take-back", "packaging", "compostable", "bio-based", "biomass", "biodegradable"
]
material_claim_pattern = re.compile(
    r"(" + "|".join(re.escape(k) for k in MATERIAL_CLAIM_WORDS) + r")",
    re.IGNORECASE
)

# section weak labels
SECTIONS = {
    "Strategy": ["strategy","ceo message","letter from","vision","roadmap","foreword","our approach","our commitment"],
    "Targets": ["target","targets","goal","goals","commitment","commitments","2030","2040","2050","science based","sbti","pathway"],
    "Environment": ["environment","climate","carbon","emissions","energy","water","waste","net zero","decarbon","biodiversity","circular"],
    "Performance": ["performance","performance data","metrics","kpi","sasb","tcfd","gri","data table","appendix"]
}

# evidence / quality patterns
RE_YEAR = re.compile(r"\b(19|20)\d{2}\b")
RE_BY_YEAR = re.compile(r"\bby\b\s*(19|20)\d{2}\b", re.IGNORECASE)
RE_PERCENT = re.compile(r"(\d+(\.\d+)?\s*%|\bpercent\b)", re.IGNORECASE)
RE_NUMBER = re.compile(r"\d")
RE_SCOPE = re.compile(r"\bscope\s*[123]\b", re.IGNORECASE)
RE_SBTI = re.compile(r"\b(sbti|science[-\s]?based)\b", re.IGNORECASE)
RE_NETZERO = re.compile(r"\bnet\s*zero\b|\bcarbon\s*neutral\b|\bcarbon\s*negative\b|\bclimate\s*positive\b|\bclimate\s*neutral\b", re.IGNORECASE)
RE_KPI = re.compile(r"\b(kpi|metric|metrics|baseline|progress|gri|sasb|tcfd)\b", re.IGNORECASE)
RE_MATERIAL = re.compile(r"\b(renewable|recycled|recyclable|rpet|fsc|certified|refillable|plant[-\s]?based|organic|bio-based|biomass|biodegradable|compostable)\b", re.IGNORECASE)
RE_CARBON = re.compile(r"\b(carbo[n]?|co2|co2e|ghg|emission|emissions|decarbon)\b", re.IGNORECASE)
RE_TARGET = re.compile(r"\b(target|goal|goals|commit|commitment|pledge|plan|roadmap|ambition|will|aim)\b", re.IGNORECASE)
RE_CIRCULAR = re.compile(r"\b(circular|recycle|recycled|recyclable|reuse|reusable|refill|second life|take back|take-back)\b", re.IGNORECASE)

# dehyphenation
RE_HYPHEN_BREAK = re.compile(r"(\w)-\s+(\w)")

# Chinese (CJK) characters
RE_CJK = re.compile(r"[\u4e00-\u9fff]")

# noise
RE_URL = re.compile(r"(https?://|www\.|\.com\b|\.org\b|\.net\b)", re.IGNORECASE)
RE_EMAIL = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
RE_PAGE_ONLY = re.compile(r"^\s*(page\s*\d+|\d+)\s*$", re.IGNORECASE)


def contains_chinese(s: str) -> bool:
    return bool(RE_CJK.search(s))


def parse_pdf_filename(filename: str):
    stem = filename.rsplit(".", 1)[0]
    parts = stem.split("_")
    if len(parts) < 2:
        return stem, "", ""
    report_year = parts[-1]

    if len(parts) >= 3:
        possible_ticker = parts[-2]
        if re.fullmatch(r"(?=.*[A-Za-z])[A-Za-z0-9]{1,10}", possible_ticker):
            return "_".join(parts[:-2]), possible_ticker.upper(), report_year

    return "_".join(parts[:-1]), "", report_year


def ensure_header(path: str, fieldnames):
    if os.path.exists(path):
        return
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writeheader()


def load_done_set(out_csv_path: str):
    done = set()
    if not os.path.exists(out_csv_path):
        return done
    try:
        with open(out_csv_path, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                folder = row.get("source_folder", "")
                file = row.get("file", "")
                if folder and file:
                    done.add((folder, file))
    except Exception:
        return set()
    return done


def is_mostly_upper(s: str, threshold: float = 0.70) -> bool:
    letters = [c for c in s if c.isalpha()]
    if not letters:
        return False
    upper_ratio = sum(c.isupper() for c in letters) / len(letters)
    return upper_ratio >= threshold


def normalize_caps_sentence(s: str) -> str:
    """
    Convert ALL-CAPS heading-like extracted text into a more standard casing.
    Keeps abbreviations like CO2 / GHG / SBTi patterns reasonably readable after cleanup.
    """
    if not is_mostly_upper(s):
        return s

    # Lower then capitalize sentence start
    s2 = s.lower().strip()
    if not s2:
        return s

    s2 = s2[0].upper() + s2[1:]

    # Restore common acronyms / ESG tokens
    replacements = {
        r"\bco2\b": "CO2",
        r"\bco2e\b": "CO2e",
        r"\bghg\b": "GHG",
        r"\bsbti\b": "SBTi",
        r"\besg\b": "ESG",
        r"\bcsr\b": "CSR",
        r"\bkpi\b": "KPI",
        r"\bprc\b": "PRC",
        r"\brspo\b": "RSPO",
        r"\bfsc\b": "FSC",
        r"\brpet\b": "rPET",
        r"\bscope 1\b": "Scope 1",
        r"\bscope 2\b": "Scope 2",
        r"\bscope 3\b": "Scope 3",
    }
    for pat, repl in replacements.items():
        s2 = re.sub(pat, repl, s2, flags=re.IGNORECASE)

    return s2


def clean_sentence(s: str) -> str:
    replacements = {
        "\u00ad": "",     # soft hyphen
        "\xa0": " ",      # non-breaking space
        "\ufeff": " ",    # BOM
        "\u200b": " ",    # zero-width space
        "\u200c": " ",
        "\u200d": " ",
        "\u2060": " ",
        "“": '"', "”": '"',
        "‘": "'", "’": "'",
        "–": "-", "—": "-", "−": "-", "-": "-", "‒": "-",
        "ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl",
        "％": "%", "／": "/", "（": "(", "）": ")", "：": ":",
    }
    for old, new in replacements.items():
        s = s.replace(old, new)

    s = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", s)
    s = RE_HYPHEN_BREAK.sub(r"\1\2", s)

    bad_chars = [
        "•", "·", "▪", "▫", "■", "□", "●", "○", "◦", "‣", "∙",
        "▶", "►", "◄", "▲", "△", "▼", "▽", "◆", "◇", "★", "☆", "※"
    ]
    for ch in bad_chars:
        s = s.replace(ch, " ")

    for ch in ["™", "®", "©"]:
        s = s.replace(ch, "")

    s = re.sub(r"[^A-Za-z0-9\s%.,:;()\[\]\"'&+\-/]", " ", s)
    s = re.sub(r"\s*/\s*", "/", s)
    s = re.sub(r"\s+", " ", s).strip()

    # normalize all-caps extracted heading-like sentences
    s = normalize_caps_sentence(s)

    return s


def is_mostly_english(s: str, threshold=0.60) -> bool:
    ss = re.sub(r"\s+", "", s)
    if not ss:
        return False
    letters = re.findall(r"[A-Za-z]", ss)
    if not letters:
        return False
    ratio = len(letters) / len(ss)
    return ratio >= threshold


def is_noise_sentence(s: str) -> bool:
    s_low = s.lower().strip()

    if not s_low:
        return True
    if RE_PAGE_ONLY.fullmatch(s_low):
        return True
    if RE_URL.search(s_low) or RE_EMAIL.search(s_low):
        return True
    if "copyright" in s_low or "all rights reserved" in s_low:
        return True
    if s_low.count("|") >= 2:
        return True
    if len(re.findall(r"[A-Za-z]", s_low)) < 5:
        return True
    if len(s_low.split()) < 3:
        return True
    if re.fullmatch(r"[\W\d_]+", s_low):
        return True

    digit_ratio = sum(ch.isdigit() for ch in s_low) / max(len(s_low), 1)
    if digit_ratio > 0.45 and not RE_PERCENT.search(s_low):
        return True

    return False


def is_table_like_row(s: str) -> bool:
    s_low = s.lower()

    year_count = len(re.findall(r"\bfy(?:19|20)\d{2}\b|\b(?:19|20)\d{2}\b", s_low))
    percent_count = len(re.findall(r"\d+(?:\.\d+)?\s*%", s_low))
    number_count = len(re.findall(r"\d+(?:,\d+)?(?:\.\d+)?", s_low))

    if year_count >= 3:
        return True
    if percent_count >= 3:
        return True
    if number_count >= 8 and len(s_low.split()) < 50:
        return True

    return False


def is_nutrition_or_product_formula_noise(s: str) -> bool:
    s_low = s.lower()

    bad_terms = [
        "dha", "choline", "selenium", "manganese", "prebiotic",
        "infant milk powder", "mama formula", "maternal mothers milk powder",
        "reduced fat", "nutritional combination", "seven key nutrients",
        "product formula", "nutrition from natural sources", "algae oil"
    ]

    return any(t in s_low for t in bad_terms)


def is_metadata_or_sdg_or_financial_noise(s: str) -> bool:
    s_low = s.lower().strip()

    header_terms = [
        "sustainability report",
        "sustainability data book",
        "non-financial report",
        "financial and non-financial highlights",
        "strategy and targets",
        "our people metric",
        "food quality and safety close up",
        "introduction fy",
        "our supply chain",
        "packaging for a better planet",
        "sustainability vision",
        "conceptual diagram",
    ]
    if any(t in s_low for t in header_terms):
        return True

    admin_terms = [
        "the materials in this report were provided",
        "may not be used for commercial purpose",
        "is the eighth non-financial report issued",
        "key performance indicator",
        "kpi (key performance indicator)",
    ]
    if any(t in s_low for t in admin_terms):
        return True

    if re.match(r"^\d+\.\d+\s+by\s+2030", s_low):
        return True
    if "sustainable development goals" in s_low:
        return True
    if "united nations sustainable development summit" in s_low:
        return True
    if "approved at the united nations" in s_low:
        return True
    if re.match(r"^\d+\s+take urgent action to combat climate change", s_low):
        return True

    finance_terms = [
        "roe", "ebitda", "dividend", "equity ratio", "free cash flow",
        "board of directors", "executive board", "base salary",
        "short-term bonus", "long-term incentives", "remuneration",
        "attendance at the board meetings", "capital expenditures",
        "net sales", "operating profit", "payout ratio", "organic growth",
        "profitability", "annual base salary", "performance rights",
        "grant value", "board year", "compensation for 2020", "shares equivalent",
        "ceo:", "executive vice president", "financial stability",
        "medium-term management plan", "long-term vision", "future creation company",
        "telecommuting program", "pension plan", "additional fees", "audit",
        "committee meetings", "board reserves", "directors believe",
        "accounting records", "inventories", "nomination and sustainability committee"
    ]
    if any(t in s_low for t in finance_terms):
        return True

    if len(re.findall(r"\b(?:19|20)\d{2}\b", s_low)) >= 3:
        return True

    if "fy201" in s_low and "%" in s_low and len(s_low.split()) < 30:
        return True

    return False


def contains_claim_content(s: str) -> bool:
    s_low = s.lower()

    has_esg = bool(kw_pattern.search(s_low))
    has_claim_cue = bool(verb_pattern.search(s_low) or phrase_pattern.search(s_low))
    has_green = bool(green_claim_pattern.search(s_low))
    has_material = bool(material_claim_pattern.search(s_low))
    has_netzero = bool(RE_NETZERO.search(s_low))
    has_quant_env = bool(
        RE_PERCENT.search(s_low) and (
            RE_CARBON.search(s_low) or
            RE_MATERIAL.search(s_low) or
            "planet" in s_low or
            "environment" in s_low or
            "water" in s_low or
            "waste" in s_low or
            "energy" in s_low
        )
    )

    if has_green or has_material or has_netzero or has_quant_env:
        return True
    if has_esg and has_claim_cue:
        return True

    return False


def section_guess_from_text(page_text: str) -> str:
    lines = [ln.strip() for ln in page_text.splitlines() if ln.strip()]
    head = " ".join(lines[:10]).lower()
    for sec in ["Strategy", "Targets", "Environment", "Performance"]:
        for w in SECTIONS.get(sec, []):
            if w in head:
                return sec
    return "Other"


def topic_guess(sentence: str) -> str:
    s = sentence.lower()
    e = any(k in s for k in E_KEYWORDS)
    so = any(k in s for k in S_KEYWORDS)
    g = any(k in s for k in G_KEYWORDS)
    if e and not so and not g:
        return "E"
    if so and not e and not g:
        return "S"
    if g and not e and not so:
        return "G"
    if e or so or g:
        return "Mixed"
    return "Other"


def claim_type_guess(s: str) -> str:
    s_low = s.lower()

    if (RE_PERCENT.search(s_low) or RE_NUMBER.search(s_low)) and (RE_BY_YEAR.search(s_low) or RE_TARGET.search(s_low)):
        return "target_commitment"
    if RE_NETZERO.search(s_low) or RE_CARBON.search(s_low):
        return "carbon_claim"
    if RE_CIRCULAR.search(s_low) or RE_MATERIAL.search(s_low) or material_claim_pattern.search(s_low):
        return "recycling_circularity"
    if green_claim_pattern.search(s_low):
        return "marketing_claim"
    if RE_PERCENT.search(s_low) or RE_NUMBER.search(s_low):
        return "quantitative_claim"
    if RE_TARGET.search(s_low):
        return "target_commitment"
    if contains_claim_content(s_low):
        return "general_green_claim"
    return "other"


def evidence_flags(s: str) -> dict:
    return {
        "has_number": int(bool(RE_NUMBER.search(s))),
        "has_year": int(bool(RE_YEAR.search(s))),
        "has_by_year": int(bool(RE_BY_YEAR.search(s))),
        "has_percent": int(bool(RE_PERCENT.search(s))),
        "has_scope": int(bool(RE_SCOPE.search(s))),
        "has_sbti": int(bool(RE_SBTI.search(s))),
        "has_netzero": int(bool(RE_NETZERO.search(s))),
        "has_kpi": int(bool(RE_KPI.search(s))),
        "has_material": int(bool(RE_MATERIAL.search(s))),
        "sentence_type": claim_type_guess(s),
        "has_green_marketing": int(bool(green_claim_pattern.search(s))),
    }


def score_sentence(s: str) -> int:
    score = 0
    s_low = s.lower()

    if contains_claim_content(s):
        score += 4

    if verb_pattern.search(s):
        score += 2
    if phrase_pattern.search(s):
        score += 2

    if green_claim_pattern.search(s):
        score += 3

    if material_claim_pattern.search(s):
        score += 2

    if RE_NUMBER.search(s):
        score += 2
    if RE_YEAR.search(s):
        score += 2
    if RE_BY_YEAR.search(s):
        score += 3
    if RE_PERCENT.search(s):
        score += 3
    if RE_SCOPE.search(s):
        score += 3
    if RE_SBTI.search(s):
        score += 3
    if RE_NETZERO.search(s):
        score += 3
    if RE_KPI.search(s):
        score += 2
    if RE_MATERIAL.search(s):
        score += 2

    ct = claim_type_guess(s)
    if ct == "target_commitment":
        score += 2
    elif ct == "quantitative_claim":
        score += 2
    elif ct == "carbon_claim":
        score += 2
    elif ct == "marketing_claim":
        score += 1
    elif ct == "recycling_circularity":
        score += 1

    if len(s) < 35:
        score -= 4
    elif len(s) < 50:
        score -= 1
    if len(s) > 380:
        score -= 2

    vague = [
        "we believe", "we strive", "we support", "our mission", "our purpose",
        "better future", "we care", "we are committed to sustainability",
        "for a better world"
    ]
    if any(v in s_low for v in vague):
        if not (
            RE_NUMBER.search(s) or RE_PERCENT.search(s) or RE_BY_YEAR.search(s)
            or green_claim_pattern.search(s)
        ):
            score -= 2

    return score


def normalize_sentence(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^\w\s%]", "", s)
    s = re.sub(r"\b(the|a|an|our|we|is|are|to|of|and|for|in)\b", " ", s)
    s = re.sub(r"\s+", "", s)
    return s


def extract_pages_pymupdf(pdf_path: str):
    pages = []
    doc = fitz.open(pdf_path)
    for i in range(doc.page_count):
        txt = doc.load_page(i).get_text("text") or ""
        pages.append((i + 1, txt))
    doc.close()
    return pages


def try_ocr(pdf_path: str) -> str:
    if not OCR_FALLBACK:
        return pdf_path
    if shutil.which("ocrmypdf") is None:
        return pdf_path

    ocr_out = pdf_path.rsplit(".", 1)[0] + "_OCR.pdf"
    if os.path.exists(ocr_out):
        return ocr_out

    try:
        cmd = ["ocrmypdf", "--skip-text", "-l", OCR_LANGUAGE, pdf_path, ocr_out]
        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if os.path.exists(ocr_out):
            return ocr_out
    except Exception:
        return pdf_path

    return pdf_path


def balanced_topk(items, k, max_per_page=2, max_per_section=5):
    out = []
    page_ct = {}
    sec_ct = {}

    for it in items:
        sc, pg, sec = it[0], it[1], it[2]
        if max_per_page is not None and page_ct.get(pg, 0) >= max_per_page:
            continue
        if max_per_section is not None and sec_ct.get(sec, 0) >= max_per_section:
            continue
        out.append(it)
        page_ct[pg] = page_ct.get(pg, 0) + 1
        sec_ct[sec] = sec_ct.get(sec, 0) + 1
        if len(out) >= k:
            break
    return out


# ---------------- Output schema ----------------
CLAIMS_FIELDS = [
    "sentence_id",
    "folder_year","sic","company","ticker","report_year",
    "source_folder","file",
    "rank","score","page","section_guess","topic_guess","sentence_type",
    "has_number","has_year","has_by_year","has_percent","has_scope","has_sbti","has_netzero","has_kpi","has_material","has_green_marketing",
    "sentence",
    "sentence_lower"
]
BAD_FIELDS = [
    "source_folder","folder_year","sic","file",
    "company","ticker","report_year",
    "char_count","status","error"
]
RUNTIME_FIELDS = [
    "source_folder","folder_year","sic","file",
    "company","ticker","report_year",
    "secs","char_count","candidate_sentences","written_topk","avg_score_topk"
]

ensure_header(OUT_CLAIMS, CLAIMS_FIELDS)
ensure_header(OUT_BAD, BAD_FIELDS)
ensure_header(OUT_RUNTIME, RUNTIME_FIELDS)

done_set = set() if FORCE_RERUN else load_done_set(OUT_CLAIMS)

# ---------------- Collect jobs ----------------
pdf_jobs = []
folders = [TEST_FOLDER] if TEST_FOLDER else sorted(os.listdir(ROOT))

for folder_name in folders:
    if not folder_name:
        continue
    folder_path = os.path.join(ROOT, folder_name)
    if not os.path.isdir(folder_path):
        continue
    m = folder_pat.match(folder_name)
    if not m:
        continue
    folder_year, sic = m.group("year"), m.group("sic")
    for fn in sorted(os.listdir(folder_path)):
        if fn.lower().endswith(".pdf"):
            if FORCE_RERUN or (folder_name, fn) not in done_set:
                pdf_jobs.append((folder_name, folder_year, sic, fn, os.path.join(folder_path, fn)))

print("Already done PDFs:", 0 if FORCE_RERUN else len(done_set))
print("Remaining PDFs:", len(pdf_jobs))

total_written = 0
sentence_counter = START_SENTENCE_ID

with open(OUT_CLAIMS, "a", newline="", encoding="utf-8-sig") as f_out, \
     open(OUT_BAD, "a", newline="", encoding="utf-8-sig") as f_bad, \
     open(OUT_RUNTIME, "a", newline="", encoding="utf-8-sig") as f_run:

    w_out = csv.DictWriter(f_out, fieldnames=CLAIMS_FIELDS)
    w_bad = csv.DictWriter(f_bad, fieldnames=BAD_FIELDS)
    w_run = csv.DictWriter(f_run, fieldnames=RUNTIME_FIELDS)

    for (folder_name, folder_year, sic, fn, pdf_path) in tqdm(pdf_jobs, desc="Processing PDFs", unit="pdf"):
        t0 = time.time()
        company, ticker, report_year = parse_pdf_filename(fn)

        pdf_to_read = try_ocr(pdf_path)

        try:
            pages = extract_pages_pymupdf(pdf_to_read)
        except Exception as e:
            w_bad.writerow({
                "source_folder": folder_name, "folder_year": folder_year, "sic": sic,
                "file": fn, "company": company, "ticker": ticker, "report_year": report_year,
                "char_count": 0, "status": "ERROR_OPEN_OR_READ", "error": str(e)[:300]
            })
            w_run.writerow({
                "source_folder": folder_name, "folder_year": folder_year, "sic": sic,
                "file": fn, "company": company, "ticker": ticker, "report_year": report_year,
                "secs": round(time.time() - t0, 2), "char_count": 0,
                "candidate_sentences": 0, "written_topk": 0, "avg_score_topk": 0
            })
            continue

        char_count = sum(len(txt) for _, txt in pages if txt)

        if char_count < MIN_CHAR_THRESHOLD:
            w_bad.writerow({
                "source_folder": folder_name, "folder_year": folder_year, "sic": sic,
                "file": fn, "company": company, "ticker": ticker, "report_year": report_year,
                "char_count": char_count, "status": "LOW_TEXT_POSSIBLE_SCAN_OR_EXTRACT_FAIL", "error": ""
            })
            w_run.writerow({
                "source_folder": folder_name, "folder_year": folder_year, "sic": sic,
                "file": fn, "company": company, "ticker": ticker, "report_year": report_year,
                "secs": round(time.time() - t0, 2), "char_count": char_count,
                "candidate_sentences": 0, "written_topk": 0, "avg_score_topk": 0
            })
            continue

        candidates = []

        # Pass 1
        for page_num, page_text in pages:
            if not page_text:
                continue
            sec = section_guess_from_text(page_text)

            for s in nltk.sent_tokenize(page_text):
                s = clean_sentence(s)
                if not s:
                    continue

                if is_noise_sentence(s):
                    continue
                if is_table_like_row(s):
                    continue
                if is_nutrition_or_product_formula_noise(s):
                    continue
                if is_metadata_or_sdg_or_financial_noise(s):
                    continue
                if REMOVE_CHINESE and contains_chinese(s):
                    continue
                if len(s) < MIN_SENT_LEN or len(s) > MAX_SENT_LEN:
                    continue
                if ENGLISH_ONLY and not is_mostly_english(s, ENGLISH_RATIO_THRESHOLD):
                    continue
                if not contains_claim_content(s):
                    continue

                tg = topic_guess(s)
                if ONLY_ENVIRONMENTAL_CLAIMS and tg not in ("E", "Mixed"):
                    continue

                sc = score_sentence(s)
                flags = evidence_flags(s)
                candidates.append((sc, page_num, sec, s, flags))

        # Pass 2
        if len(candidates) < TOP_K:
            relaxed_candidates = []
            for page_num, page_text in pages:
                if not page_text:
                    continue
                sec = section_guess_from_text(page_text)

                for s in nltk.sent_tokenize(page_text):
                    s = clean_sentence(s)
                    if not s:
                        continue

                    if is_noise_sentence(s):
                        continue
                    if is_table_like_row(s):
                        continue
                    if is_nutrition_or_product_formula_noise(s):
                        continue
                    if is_metadata_or_sdg_or_financial_noise(s):
                        continue
                    if REMOVE_CHINESE and contains_chinese(s):
                        continue
                    if len(s) < MIN_SENT_LEN or len(s) > MAX_SENT_LEN:
                        continue
                    if ENGLISH_ONLY and not is_mostly_english(s, ENGLISH_RATIO_THRESHOLD):
                        continue

                    relaxed_hit = (
                        kw_pattern.search(s) or
                        green_claim_pattern.search(s) or
                        material_claim_pattern.search(s)
                    )
                    if not relaxed_hit:
                        continue

                    tg = topic_guess(s)
                    if ONLY_ENVIRONMENTAL_CLAIMS and tg not in ("E", "Mixed"):
                        continue

                    sc = score_sentence(s) - 1
                    flags = evidence_flags(s)
                    relaxed_candidates.append((sc, page_num, sec, s, flags))

            if relaxed_candidates:
                candidates.extend(relaxed_candidates)

        # Deduplicate
        best = {}
        for sc, pg, sec, sent, flags in candidates:
            norm = normalize_sentence(sent)
            if norm not in best or sc > best[norm][0]:
                best[norm] = (sc, pg, sec, sent, flags)

        uniq = list(best.values())
        uniq.sort(key=lambda x: x[0], reverse=True)

        # Top-k
        if PAGE_BALANCE:
            top = balanced_topk(
                [(sc, pg, sec, sent, flags) for (sc, pg, sec, sent, flags) in uniq],
                TOP_K,
                max_per_page=MAX_PER_PAGE,
                max_per_section=MAX_PER_SECTION
            )
        else:
            top = [(sc, pg, sec, sent, flags) for (sc, pg, sec, sent, flags) in uniq[:TOP_K]]

        for r, (sc, pg, sec, sent, flags) in enumerate(top, start=1):
            tg = topic_guess(sent)
            row = {
                "sentence_id": f"sen_{sentence_counter}",
                "folder_year": folder_year,
                "sic": sic,
                "company": company,
                "ticker": ticker,
                "report_year": report_year,
                "source_folder": folder_name,
                "file": fn,
                "rank": r,
                "score": sc,
                "page": pg,
                "section_guess": sec,
                "topic_guess": tg,
                "sentence_type": flags["sentence_type"],
                "has_number": flags["has_number"],
                "has_year": flags["has_year"],
                "has_by_year": flags["has_by_year"],
                "has_percent": flags["has_percent"],
                "has_scope": flags["has_scope"],
                "has_sbti": flags["has_sbti"],
                "has_netzero": flags["has_netzero"],
                "has_kpi": flags["has_kpi"],
                "has_material": flags["has_material"],
                "has_green_marketing": flags["has_green_marketing"],
                "sentence": sent,
                "sentence_lower": sent.lower()
            }
            w_out.writerow(row)
            total_written += 1
            sentence_counter += 1

        avg_score_topk = round(sum(x[0] for x in top) / len(top), 2) if top else 0

        w_run.writerow({
            "source_folder": folder_name, "folder_year": folder_year, "sic": sic,
            "file": fn, "company": company, "ticker": ticker, "report_year": report_year,
            "secs": round(time.time() - t0, 2),
            "char_count": char_count,
            "candidate_sentences": len(uniq),
            "written_topk": len(top),
            "avg_score_topk": avg_score_topk
        })

print("DONE ✅  New sentences written:", total_written)
print("Outputs:", OUT_CLAIMS, OUT_BAD, OUT_RUNTIME)

Already done PDFs: 0
Remaining PDFs: 237


Processing PDFs: 100%|██████████| 237/237 [05:15<00:00,  1.33s/pdf]

DONE ✅  New sentences written: 472
Outputs: claims_research.csv bad_pdfs.csv runtime_report.csv


從企業永續報告 PDF 中，自動抽出包含環境永續宣稱的高品質完整句子，並透過規則過濾、Sentence 偵測、特徵標記、分數排序與去重後，輸出成研究可用的 Sentence-level CSV 資料集。